In [10]:
# ============================================================
# Batch LLM coding of open-ended odd-one-out strategies
# Multi-Experiment Runner (Exp 1 & Exp 2)
# ============================================================
# Input:
#   Open_Answers/Curvature Open Answers.xlsx
#   Sheets:
#     - 'exp1 curv'    (Features: smoothness, vertices, angle, intuition, globalness, size)
#     - 'exp1 noInstr' (Features: smoothness, vertices, angle, intuition, globalness, size)
#     - 'exp2 curv'    (Features: smoothness, vertices, range, intuition, globalness, animal-like, size)
#     - 'exp2 noInstr' (Features: smoothness, vertices, range, intuition, globalness, animal-like, size)
#
# Output:
#   Open_Answers/Curvature_Open_Answers_LLM_coded_{PROMPT_STYLE}.xlsx
#   (e.g., Curvature_Open_Answers_LLM_coded_minimal.xlsx or
#          Curvature_Open_Answers_LLM_coded_detailed.xlsx)
#   All 4 experiments are saved in separate tabs within this single workbook.
#
# Key Features:
#   - Resumable: Reads existing output file if present; skips already completed participants.
#   - Dynamic Output Filename: Automatically reflects the selected PROMPT_STYLE.
#   - Experiment-specific prompts:
#       * Exp 1 features 'angle' (difference in degrees of angle).
#       * Exp 2 features 'range' (how deep/big the curves/angles can be) and 'animal-like'.
#   - Saves progress after every batch to prevent loss.
#
# Requirements:
#   pip install -U requests pandas openpyxl python-dotenv
# ============================================================
import os
import json
import random
import time
import pandas as pd
import requests
from dotenv import load_dotenv


In [11]:
# ------------------------------------------------------------
# Verify Groq API connection and available models
# ------------------------------------------------------------
load_dotenv()
load_dotenv(".env")
load_dotenv("Open_Answers/.env")

groq_key = os.environ.get("GROQ_API_KEY")
if not groq_key:
    print("WARNING: GROQ_API_KEY not found in environment or .env. Please set it before running batch coding.")
else:
    url = "https://api.groq.com/openai/v1/models"
    try:
        r = requests.get(
            url,
            headers={"Authorization": f"Bearer {groq_key}"},
            timeout=15
        )
        print("API Status:", r.status_code)
        if r.status_code == 200:
            models = r.json().get("data", [])
            print(f"Connection successful. {len(models)} models available.")
        else:
            print("API Response:", r.text[:300])
    except Exception as e:
        print(f"Could not connect to Groq API: {e}")


API Status: 200
Connection successful. 11 models available.


In [12]:
# ------------------------------------------------------------
# 1. Setup & Experiment Configuration
# ------------------------------------------------------------
# Ensure file paths always resolve to the 'Open_Answers' folder
cwd = os.getcwd()
if os.path.basename(cwd) == "Open_Answers":
    OPEN_ANSWERS_DIR = cwd
elif os.path.exists(os.path.join(cwd, "Open_Answers")):
    OPEN_ANSWERS_DIR = os.path.join(cwd, "Open_Answers")
else:
    OPEN_ANSWERS_DIR = cwd

# Select active prompt style: 'minimal' or 'detailed'
PROMPT_STYLE = "detailed"

INPUT_FILE = os.path.join(OPEN_ANSWERS_DIR, "Curvature Open Answers.xlsx")

# Output filename dynamically reflects the chosen prompt_style
OUTPUT_FILE = os.path.join(OPEN_ANSWERS_DIR, f"Curvature_Open_Answers_LLM_coded_{PROMPT_STYLE}.xlsx")

MODEL = "openai/gpt-oss-120b"
BATCH_SIZE = 5
API_URL = "https://api.groq.com/openai/v1/chat/completions"

# Configurations for each of the 4 experiment sheets:
# - Exp 1 experiments have 'angle' as one of the features
# - Exp 2 experiments have 'range' and 'animal-like' as features
# The column ordering aligns with the standard OOO analysis structure.
EXPERIMENT_CONFIGS = {
    "exp1 curv": {
        "exp_type": "exp1",
        "text_col": "strategy_open",
        "feature_cols": [
            "smoothness",
            "vertices",
            "angle",
            "intuition",
            "globalness",
            "size"
        ]
    },
    "exp1 noInstr": {
        "exp_type": "exp1",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "angle",
            "intuition",
            "globalness",
            "size"
        ]
    },
    "exp2 curv": {
        "exp_type": "exp2",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "range",
            "intuition",
            "globalness",
            "animal-like",
            "size"
        ]
    },
    "exp2 noInstr": {
        "exp_type": "exp2",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "range",
            "intuition",
            "globalness",
            "animal-like",
            "size"
        ]
    }
}

print(f"Selected prompt style: [{PROMPT_STYLE}]")
print(f"Input file:            {INPUT_FILE}")
print(f"Output file:           {OUTPUT_FILE}")
print(f"Configured experiments: {list(EXPERIMENT_CONFIGS.keys())}")


Selected prompt style: [detailed]
Input file:            /Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature Open Answers.xlsx
Output file:           /Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx
Configured experiments: ['exp1 curv', 'exp1 noInstr', 'exp2 curv', 'exp2 noInstr']


In [13]:
# ------------------------------------------------------------
# 2. LLM coding prompts (Exp 1 vs Exp 2)
# ------------------------------------------------------------

PROMPT_EXP1_DETAILED = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "angle": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions:
- smoothness: curvature, sharpness, pointedness, jaggedness, roundness, spikiness, bumpiness, or local contour quality. (Do not code corners or points as smoothness if the participant means how many there are).
- vertices: count, quantity, presence/absence, or comparison of vertices, points, corners, tips, sides, bumps, lumps, or edges. (Do not code this if the participant means sharpness/curvature rather than quantity).
- angle: difference in degrees of angle, steepness, slope, tilt, or inclination of a line, edge, bump, curve, or corner.
- intuition: feeling, instinct, gut reaction, immediate impression, guess, vague sense, or non-explicit decision process.
- globalness: overall shape, global configuration, holistic appearance, global structure, overall pattern, or general impression of the whole stimulus rather than a local feature.
- size: overall size, magnitude, area, length, height, width, largeness, smallness, or physical extent.

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature, even if other features are also mentioned.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

PROMPT_EXP1_MINIMAL = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "angle": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions:
- smoothness: If the explanation is related to sharpness or smoothness / curvature
- vertices: If the explanation is related to the count of vertices, points, corners, or edges
- angle: If the explanation is related to angle (difference in degrees of angle, steepness, slope)
- intuition: Gut feeling, instinct, or intuition
- globalness: Overall shape or global configuration
- size: Size of the shape (largeness, smallness, extent)

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

PROMPT_EXP2_DETAILED = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "range": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "animal-like": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions:
- smoothness: curvature, sharpness, pointedness, jaggedness, roundness, spikiness, bumpiness, or local contour quality. (Do not code corners or points as smoothness if the participant means how many there are).
- vertices: count, quantity, presence/absence, or comparison of vertices, points, corners, tips, sides, bumps, lumps, or edges. (Do not code this if the participant means sharpness/curvature rather than quantity).
- range: how deep or big the curves or angles can be (depth of curves/angles, amplitude, how deep/pronounced or extreme the bumps/curves are).
- intuition: feeling, instinct, gut reaction, immediate impression, guess, vague sense, or non-explicit decision process.
- globalness: overall shape, global configuration, holistic appearance, global structure, overall pattern, or general impression of the whole stimulus rather than a local feature.
- animal-like: when participants mention that certain shapes look like animals, animal parts, organisms, creatures, or recognizable animal-like figures.
- size: overall size, magnitude, area, length, height, width, largeness, smallness, or physical extent.

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature, even if other features are also mentioned.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

PROMPT_EXP2_MINIMAL = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "range": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "animal-like": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions:
- smoothness: If the explanation is related to sharpness or smoothness / curvature
- vertices: If the explanation is related to the count of vertices, points, corners, or edges
- range: If the explanation is related to range (how deep or big the curves or angles can be)
- intuition: Gut feeling, instinct, or intuition
- globalness: Overall shape or global configuration
- animal-like: When people mention that certain shapes look like animals or creatures
- size: Size of the shape (largeness, smallness, extent)

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

# You can also change the prompt style here; doing so automatically syncs OUTPUT_FILE
def set_prompt_style(style: str):
    """Update active prompt style and dynamically refresh OUTPUT_FILE path."""
    global PROMPT_STYLE, OUTPUT_FILE
    if style not in ("minimal", "detailed"):
        print(f"Note: Using custom prompt style '{style}'.")
    PROMPT_STYLE = style
    OUTPUT_FILE = os.path.join(OPEN_ANSWERS_DIR, f"Curvature_Open_Answers_LLM_coded_{PROMPT_STYLE}.xlsx")
    print(f"Prompt style set to: [{PROMPT_STYLE}]")
    print(f"Output file set to:   [{OUTPUT_FILE}]")

# Initialize or re-sync with current PROMPT_STYLE
set_prompt_style(PROMPT_STYLE)

def get_prompt(exp_type: str, style: str = None) -> str:
    """Return the prompt corresponding to experiment type and style."""
    active_style = style if style is not None else PROMPT_STYLE
    if exp_type == "exp1":
        return PROMPT_EXP1_DETAILED if active_style == "detailed" else PROMPT_EXP1_MINIMAL
    elif exp_type == "exp2":
        return PROMPT_EXP2_DETAILED if active_style == "detailed" else PROMPT_EXP2_MINIMAL
    else:
        raise ValueError(f"Unknown experiment type: {exp_type}")


Prompt style set to: [detailed]
Output file set to:   [/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx]


In [14]:
# ------------------------------------------------------------
# 3. Dynamic JSON schema generator for structured output
# ------------------------------------------------------------
def get_response_schema(feature_cols: list[str]) -> dict:
    """Generate a strict JSON schema for validating LLM structured output."""
    props = {col: {"type": "integer", "enum": [0, 1]} for col in feature_cols}
    props.update({
        "item_id": {"type": "string"},
        "mentioned_terms": {"type": "array", "items": {"type": "string"}},
        "rationale": {"type": "string"},
        "uncertain": {"type": "boolean"}
    })
    return {
        "type": "object",
        "properties": {
            "results": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": props,
                    "required": ["item_id", *feature_cols, "mentioned_terms", "rationale", "uncertain"]
                }
            }
        },
        "required": ["results"]
    }


In [15]:
# ------------------------------------------------------------
# 4. Helper functions for batch LLM coding
# ------------------------------------------------------------
def make_error_row(feature_cols: list[str], error_message: str) -> dict:
    """Create a standardized failed-coding row."""
    return {
        **{feat: None for feat in feature_cols},
        "mentioned_terms": [],
        "rationale": f"API error: {error_message}",
        "uncertain": True
    }


def code_batch(
    batch_df: pd.DataFrame,
    text_col: str,
    feature_cols: list[str],
    prompt: str,
    max_attempts: int = 4
) -> list[dict]:
    """
    Code a batch of participant responses in one Groq API request.
    Handles empty responses cleanly without wasting API calls.
    """
    coded_rows = []
    non_empty_indices = []

    for idx, row in batch_df.iterrows():
        resp_text = str(row[text_col]).strip() if pd.notna(row[text_col]) else ""
        if not resp_text or resp_text.lower() in ("nan", "none", ""):
            # Participant left response blank
            coded_rows.append({
                "subj_id": str(row["subj_id"]),
                text_col: "",
                **{feat: 0 for feat in feature_cols},
                "mentioned_terms": [],
                "rationale": "No response provided",
                "uncertain": True
            })
        else:
            non_empty_indices.append(idx)

    if not non_empty_indices:
        return coded_rows

    active_batch = batch_df.loc[non_empty_indices]

    items = [
        {
            "item_id": str(row["subj_id"]),
            "response": str(row[text_col]).strip()
        }
        for _, row in active_batch.iterrows()
    ]

    expected_ids = {str(x) for x in active_batch["subj_id"]}
    payload_text = json.dumps(items, ensure_ascii=False)

    api_key = os.environ.get("GROQ_API_KEY", "")
    if not api_key:
        raise ValueError("GROQ_API_KEY environment variable is not set.")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    request_body = {
        "model": MODEL,
        "temperature": 0,
        "max_completion_tokens": 8000,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": prompt},
            {"role": "user", "content": payload_text}
        ]
    }

    last_error = None

    for attempt in range(1, max_attempts + 1):
        print(f"Groq attempt {attempt}/{max_attempts}...", flush=True)

        try:
            response = requests.post(
                API_URL,
                headers=headers,
                json=request_body,
                timeout=120
            )
            response.raise_for_status()

            content = response.json()["choices"][0]["message"]["content"]
            parsed = json.loads(content)

            # Accept {"results": [...]}, direct [...], or single {...}
            if isinstance(parsed, dict) and "results" in parsed:
                results = parsed["results"]
            elif isinstance(parsed, list):
                results = parsed
            elif isinstance(parsed, dict) and "item_id" in parsed:
                results = [parsed]
            else:
                raise ValueError(
                    f"Model returned unexpected JSON structure: {parsed!r}"
                )

            if not isinstance(results, list):
                raise ValueError("Model did not return a list of coded results.")

            returned_ids = {str(r.get("item_id")) for r in results}
            if returned_ids != expected_ids:
                raise ValueError(
                    f"ID mismatch. Expected: {sorted(expected_ids)}, Received: {sorted(returned_ids)}"
                )

            for result in results:
                # Validate feature codes
                for feature in feature_cols:
                    value = result.get(feature)
                    if type(value) is not int or value not in (0, 1):
                        raise ValueError(
                            f"Invalid code for {feature} in item {result.get('item_id')}: {value!r}"
                        )

                if type(result.get("uncertain")) is not bool:
                    raise ValueError("uncertain must be a boolean.")
                if not isinstance(result.get("mentioned_terms"), list):
                    raise ValueError("mentioned_terms must be a list.")
                if not isinstance(result.get("rationale"), str):
                    raise ValueError("rationale must be a string.")

                original = active_batch[
                    active_batch["subj_id"].astype(str) == str(result["item_id"])
                ].iloc[0]

                coded_rows.append({
                    "subj_id": str(original["subj_id"]),
                    text_col: original[text_col],
                    **{col: result[col] for col in feature_cols},
                    "mentioned_terms": "; ".join(result["mentioned_terms"]) if isinstance(result["mentioned_terms"], list) else str(result["mentioned_terms"]),
                    "rationale": result["rationale"],
                    "uncertain": result["uncertain"]
                })

            return coded_rows

        except (requests.RequestException, json.JSONDecodeError, ValueError, KeyError) as exc:
            last_error = exc
            print(f"Attempt {attempt} failed: {repr(exc)}", flush=True)

            if attempt == max_attempts:
                raise

            wait_seconds = min(15 * (2 ** (attempt - 1)), 240) + random.uniform(0, 5)
            print(f"Waiting {wait_seconds:.0f} seconds...", flush=True)
            time.sleep(wait_seconds)

    raise last_error


In [16]:
# ------------------------------------------------------------
# 5. Load data and run batch coding across all 4 experiments
# ------------------------------------------------------------
print(f"Active Prompt Style: [{PROMPT_STYLE}]")
print(f"Target Output File:  [{OUTPUT_FILE}]\n")

# Load all existing sheets from OUTPUT_FILE if it exists (for resumability)
existing_sheets = {}
if os.path.exists(OUTPUT_FILE):
    try:
        xl_existing = pd.ExcelFile(OUTPUT_FILE)
        for sheet_name in xl_existing.sheet_names:
            existing_sheets[sheet_name] = pd.read_excel(OUTPUT_FILE, sheet_name=sheet_name)
        print(f"Existing output workbook found with sheets: {list(existing_sheets.keys())}")
    except Exception as e:
        print(f"Could not read existing output workbook: {e}")

xl_input = pd.ExcelFile(INPUT_FILE)

for sheet_name, cfg in EXPERIMENT_CONFIGS.items():
    print("\n" + "=" * 60)
    print(f"PROCESSING EXPERIMENT: {sheet_name}")
    print("=" * 60)

    if sheet_name not in xl_input.sheet_names:
        print(f"Sheet '{sheet_name}' not found in input file! Skipping.")
        continue

    exp_type = cfg["exp_type"]
    text_col = cfg["text_col"]
    feature_cols = cfg["feature_cols"]
    prompt = get_prompt(exp_type)

    df_raw = pd.read_excel(INPUT_FILE, sheet_name=sheet_name)
    df_raw.columns = df_raw.columns.str.strip()

    # Identify text column fallback if needed
    if text_col not in df_raw.columns:
        if "open_answer" in df_raw.columns:
            text_col = "open_answer"
        elif "strategy_open" in df_raw.columns:
            text_col = "strategy_open"
        else:
            raise ValueError(f"Could not find response text column in {sheet_name}")

    # Clean participant IDs and text
    df_raw["subj_id"] = df_raw["subj_id"].astype(str).str.strip()
    df_raw[text_col] = df_raw[text_col].fillna("").astype(str).str.strip()

    print(f"Total rows in sheet: {len(df_raw)}")

    # Retrieve or initialize sheet dataframe
    if sheet_name in existing_sheets and "subj_id" in existing_sheets[sheet_name].columns:
        out = existing_sheets[sheet_name]
        out["subj_id"] = out["subj_id"].astype(str).str.strip()
        print(f"Found existing output for '{sheet_name}': {len(out)} rows")
    else:
        out = pd.DataFrame(columns=[
            "subj_id", text_col, *feature_cols,
            "mentioned_terms", "rationale", "uncertain"
        ])

    # Check which participants already have valid feature codes
    complete_ids = set()
    if not out.empty and all(col in out.columns for col in feature_cols):
        complete_ids = set(out.dropna(subset=feature_cols)["subj_id"].astype(str))

    todo = df_raw[~df_raw["subj_id"].isin(complete_ids)].copy()
    print(f"Already coded: {len(complete_ids)} | Needing coding: {len(todo)}")

    if len(todo) == 0:
        print(f"All participants in '{sheet_name}' already coded.")
        existing_sheets[sheet_name] = out
        continue

    # Process batches
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo.iloc[start:start + BATCH_SIZE]
        batch_num = start // BATCH_SIZE + 1
        total_batches = (len(todo) - 1) // BATCH_SIZE + 1

        print(
            f"\n[{sheet_name}] Batch {batch_num}/{total_batches} "
            f"({len(batch)} responses)...",
            flush=True
        )

        try:
            batch_coded = code_batch(batch, text_col, feature_cols, prompt)
        except Exception as e:
            print(f"Batch failed: {e}", flush=True)
            batch_coded = [
                {
                    "subj_id": str(row["subj_id"]),
                    text_col: row[text_col],
                    **make_error_row(feature_cols, str(e))
                }
                for _, row in batch.iterrows()
            ]

        # Remove previous incomplete rows for these IDs and append new
        batch_ids = {str(x) for x in batch["subj_id"]}
        if not out.empty and "subj_id" in out.columns:
            out = out[~out["subj_id"].astype(str).isin(batch_ids)]

        out = pd.concat([out, pd.DataFrame(batch_coded)], ignore_index=True)

        # Standardize column order matching OOO analysis structure
        col_order = ["subj_id", text_col, *feature_cols, "mentioned_terms", "rationale", "uncertain"]
        out = out.reindex(columns=[c for c in col_order if c in out.columns] + [c for c in out.columns if c not in col_order])

        existing_sheets[sheet_name] = out

        # Save all sheets to OUTPUT_FILE after every batch
        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
            for s_name, s_df in existing_sheets.items():
                s_df.to_excel(writer, sheet_name=s_name, index=False)

        print(f"Saved progress to '{OUTPUT_FILE}' ({sheet_name}: {len(out)} rows).")
        time.sleep(2)

print("\nAll experiments processed.")


Active Prompt Style: [detailed]
Target Output File:  [/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx]


PROCESSING EXPERIMENT: exp1 curv
Total rows in sheet: 35
Already coded: 0 | Needing coding: 35

[exp1 curv] Batch 1/7 (5 responses)...
Groq attempt 1/4...
Saved progress to '/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx' (exp1 curv: 5 rows).

[exp1 curv] Batch 2/7 (5 responses)...
Groq attempt 1/4...
Saved progress to '/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx' (exp1 curv: 10 rows).

[exp1 curv] Batch 3/7 (5 responses)...
Groq attempt 1/4...
Saved progress to '/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx' (exp1 curv: 15 rows).

[exp1 curv] Batch 4/7 (5 responses)...
Groq attempt 1/4...

In [17]:
# ------------------------------------------------------------
# 6. Brief summary across all experiments
# ------------------------------------------------------------
# First, ensure existing_sheets in memory is saved cleanly to OUTPUT_FILE
# (recovers the file if a previous write was interrupted)
if "existing_sheets" in globals() and existing_sheets:
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        for s_name, s_df in existing_sheets.items():
            cfg = EXPERIMENT_CONFIGS.get(s_name)
            if cfg:
                for col in cfg["feature_cols"]:
                    if col in s_df.columns:
                        s_df[col] = pd.to_numeric(s_df[col], errors="coerce").astype("Int64")
            s_df.to_excel(writer, sheet_name=s_name, index=False)
    print(f"Saved clean workbook: {OUTPUT_FILE}\n")

# Load all sheets into memory BEFORE any processing (never read inside ExcelWriter!)
sheets_to_summarize = {}
if os.path.exists(OUTPUT_FILE):
    try:
        sheets_to_summarize = pd.read_excel(OUTPUT_FILE, sheet_name=None, engine="openpyxl")
    except Exception as e:
        print(f"Could not read {OUTPUT_FILE} from disk ({e}). Using in-memory data if available.")
        if "existing_sheets" in globals():
            sheets_to_summarize = existing_sheets

if sheets_to_summarize:
    summary_rows = []
    for sheet_name, df_sheet in sheets_to_summarize.items():
        cfg = EXPERIMENT_CONFIGS.get(sheet_name, {})
        feat_cols = [c for c in cfg.get("feature_cols", []) if c in df_sheet.columns]
        total_rows = len(df_sheet)
        coded_rows = int(df_sheet.dropna(subset=feat_cols).shape[0]) if feat_cols else 0
        errors = int(df_sheet["rationale"].astype(str).str.startswith("API_ERROR").sum()) if "rationale" in df_sheet.columns else 0
        uncertain = int(pd.to_numeric(df_sheet["uncertain"], errors="coerce").fillna(0).sum()) if "uncertain" in df_sheet.columns else 0
        
        summary_rows.append({
            "Experiment": sheet_name,
            "Status": "✓ Complete" if (coded_rows == total_rows and errors == 0) else "⚠ Check",
            "Total Rows": total_rows,
            "Coded": coded_rows,
            "API Errors": errors,
            "Uncertain": uncertain
        })
    
    summary_df = pd.DataFrame(summary_rows)
    print("=== Brief Experiment Summary ===")
    print(summary_df.to_string(index=False))
else:
    print(f"No valid output data found for {OUTPUT_FILE}.")


Saved clean workbook: /Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_detailed.xlsx

=== Brief Experiment Summary ===
  Experiment     Status  Total Rows  Coded  API Errors  Uncertain
   exp1 curv ✓ Complete          35     35           0          1
exp1 noInstr ✓ Complete          36     36           0          1
   exp2 curv ✓ Complete          36     36           0          6
exp2 noInstr ✓ Complete          36     36           0          1
